# Objective

The objective of this project is to build a robust generative search system that can accurately interpret and answer questions derived from a collection of policy documents. Using frameworks such as LangChain, the system will combine document parsing, embedding-based retrieval, and LLM-powered generation to deliver reliable, context-aware responses. By the end of this project, we aim to create an end-to-end pipeline capable of:

- Efficiently ingesting and indexing policy documents

- Retrieving highly relevant context using vector-based search

- Generating precise, grounded answers with minimal hallucination

- Ensuring scalability, extensibility, and ease of integration for future enhancements

### Why LangChain ?
### For this project we will be using langchain framework due to its widespread adopation for building Gen AI based application because it packs with most of the funtionalities required to build applications and now it has the stabel 1.0 version so there won't be breaking changing until the next major release. these are some specific reason for choosing a frame work like langchain are :-

- We can swap in and out various building blocks of RAG applications like AI model, Vector DB, retriever mechanism to compare what best fit our application without disturbing other elements of app.
- As Langchain has been around for a while now so there is a strong community support to find solutions easily
- One stop place to install, plugging in the various tools and functionality 
- Though we have not implemented the tracing in this project but it can be implemented easily in the future without have to shift to other framework make it ecosystem friendly for future advancements

## Installing important libraries

In [1]:
%pip install openai langchain faiss-cpu pypdf tiktoken docarray PyPDF tiktoken langchain-openai flashrank langchain-community pillow sentence-transformers langchain-docling accelerate arize-phoenix
# langchain-docling

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 37.5 MB/s  0:00:00
   ---------------------------------------- 0.0/804.3 kB ? eta -:--:--
   ---------------------------------------- 804.3/804.3 kB 29.7 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 33.0 MB/s  0:00:00
   ---------------------------------------- 0.0/4.7 MB ? eta -:--:--
   ---------------------------------------- 4.7/4.7 MB 58.9 MB/s  0:00:00
   ---------------------------------------- 0.0/28.0 MB ? eta -:--:--
   ----------------- ---------------------- 12.3/28.0 MB 64.5 MB/s eta 0:00:01
   ----------------------------------- ---- 24.6/28.0 MB 59.7 MB/s eta 0:00:01
   ---------------------------------------- 28.0/28.0 MB 54.0 MB/s  0:00:00

    ---------------------------------------  1/42 [zipp]
   -- -------------------------------------  3/42 [sqlean-py


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importing important libraries and functionalities 

In [ ]:
from dotenv import load_dotenv

import os
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.agents.middleware import PIIMiddleware
import re
from typing import List
# from langchain_docling import DoclingLoader


In [3]:
load_dotenv()

True

In [5]:
from langsmith import Client
client = Client()
prompt = client.pull_prompt("insurance-policy-system-prompt:efd03fd9")

## loading and parsing insurance policy PDF documents

In [6]:
import glob
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

pdf_files = glob.glob("Policy+Documents/*.pdf")

loader = DoclingLoader(
    file_path=pdf_files,
    export_type=ExportType.DOC_CHUNKS
)

documents = loader.load()

2025-11-20 14:48:50,634 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-20 14:48:51,619 - INFO - Going to convert document batch...
2025-11-20 14:48:51,620 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-20 14:48:51,657 - INFO - Loading plugin 'docling_defaults'
2025-11-20 14:48:51,672 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-11-20 14:48:51,672 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-20 14:48:51,691 - INFO - Loading plugin 'docling_defaults'
2025-11-20 14:48:51,731 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-11-20 14:48:51,731 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-20 14:48:52,977 - INFO - Accelerator device: 'cpu'
[INFO] 2025-11-20 1

In [7]:

# Load PDF documents with error handling
# print("Loading PDF documents from ./Policy+Documents...")
# try:
#     pdf_directory_loader = PyPDFDirectoryLoader("./Policy+Documents")
#     documents = pdf_directory_loader.load()
#     print(f"✓ Successfully loaded {len(documents)} documents")
#     print(f"✓ Total pages: {sum(doc.metadata.get('total_pages', 1) for doc in documents)}")
# except Exception as e:
#     print(f"Error loading documents: {str(e)}")
#     raise

In [8]:
documents[0].page_content[:100]

"- <<Date>>\n- <<Policyholder's Name>>\n- <<Policyholder's Address>>\n- <<Policyholder's Contact Number>"

## Splitting the documents in chunks for efficient storage and retrival for our vector store

In [9]:
# Split documents into chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(documents)
print(f"✓ Created {len(splits)} document chunks")
print(f"✓ Average chunk size: {sum(len(s.page_content) for s in splits) // len(splits)} characters")

Splitting documents into chunks...
✓ Created 1136 document chunks
✓ Average chunk size: 565 characters


In [10]:
print(splits[0])

page_content='- <<Date>>
- <<Policyholder's Name>>
- <<Policyholder's Address>>
- <<Policyholder's Contact Number>>
Dear <<Policyholder's Name>>,' metadata={'source': 'Policy+Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/0', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 1, 'bbox': {'l': 72.0, 't': 766.5540649804688, 'r': 115.926, 'b': 758.3620649804687, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 8]}]}, {'self_ref': '#/texts/1', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 1, 'bbox': {'l': 72.0, 't': 755.0340649804688, 'r': 182.527, 'b': 746.8420649804688, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 23]}]}, {'self_ref': '#/texts/2', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'cont

## Initializing the Embedding model for using it to create embeddings that to be stored in our vectorDB 

In [ ]:
# Initialize embeddings model with timeout and retry settings
print("Initializing OpenAI embeddings model...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",  # Using smaller, faster model
    request_timeout=60,  # 60 second timeout
    max_retries=3  # Retry up to 3 times on failure
)
print("✓ Embeddings model initialized")

Initializing OpenAI embeddings model...
✓ Embeddings model initialized


In [12]:
# Test embedding on a single document
print("Testing embeddings on a sample chunk...")
try:
    test_embedding = embeddings_model.embed_documents([splits[0].page_content])
    print(f"✓ Test embedding successful - dimension: {len(test_embedding[0])}")
except Exception as e:
    print(f"✗ Error testing embeddings: {str(e)}")
    raise

Testing embeddings on a sample chunk...


2025-11-20 15:01:15,562 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


✓ Test embedding successful - dimension: 1536


#### Initializing cache backed embeddings as it makes our app more efficient by caching the already computed embedding hence saving cost and latency hence improving our application performance leading to rich user experience

In [13]:
from langchain_classic.embeddings import CacheBackedEmbeddings  
from langchain_classic.storage import LocalFileStore 
store = LocalFileStore("./cache/") 

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model,
    store,
    namespace="semantic-spotter"
)

c:\Users\rocky\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [14]:
# Preview first few splits (safe check)
print("Preview of document splits:")
try:
    for i, split in enumerate(splits[:3]):
        print(f"\n--- Split {i+1} ---")
        print(f"Source: {split.metadata.get('source', 'Unknown')}")
        print(f"Page: {split.metadata.get('page', 'Unknown')}")
        print(f"Content preview: {split.page_content[:150]}...")
    print(f"\n✓ Total splits available: {len(splits)}")
except Exception as e:
    print(f"Error previewing splits: {e}")
    raise

Preview of document splits:

--- Split 1 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: - <<Date>>
- <<Policyholder's Name>>
- <<Policyholder's Address>>
- <<Policyholder's Contact Number>>
Dear <<Policyholder's Name>>,...

--- Split 2 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: Sub: Your Policy no. <<  >>
We are glad to inform you that your proposal has been accepted and the HDFC Life Easy Health ('Policy') being this documen...

--- Split 3 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: Policy document:
As an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy is  enclosed herewith. Pleas...

✓ Total splits available: 1136


#### Computing, storing and loading from/to our vectorDB based on if the embedding are computed and stored already.

In [15]:

def create_vector_store_faiss(splits, embeddings_model, save_path="./faiss_store"):
    """Create and save FAISS vector store."""
    print(f"Creating vector store from {len(splits)} documents...")
    start_time = time.time()
    
    try:
        # Create FAISS store directly from documents
        if os.path.exists(save_path):
            return FAISS.load_local(save_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)
        
        vectordb = FAISS.from_documents(
            documents=splits,
            embedding=embeddings_model
        )
        print(f"✓ FAISS vector store created")
        
        # Save to disk
        os.makedirs(save_path, exist_ok=True)
        vectordb.save_local(save_path)
        print(f"✓ Saved to: {save_path}")
        
        elapsed = time.time() - start_time
        print(f"✓ Time: {elapsed:.1f}s ({elapsed/60:.1f}m)")
        
        return vectordb
    except Exception as e:
        print(f"✗ Error: {type(e).__name__}: {e}")
        raise


In [16]:
# Create the vector store
try:
    vectordb = create_vector_store_faiss(splits, cached_embedder, "./faiss_store")
    print("✓ Vector store ready for similarity search")
except Exception as e:
    print(f"Failed to create vector store: {str(e)}")
    raise


2025-11-20 15:01:15,701 - INFO - Loading faiss with AVX2 support.


Creating vector store from 1136 documents...


2025-11-20 15:01:15,821 - INFO - Successfully loaded faiss with AVX2 support.


✓ Vector store ready for similarity search


In [17]:
#### Implementing Compression and reranker to improve our retrival relancy/performance further without adding much latency by using lightweight reranker

In [18]:
from langchain.tools import tool
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank()
compression_retriever = None

if 'vectordb' in globals() and vectordb is not None:
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=vectordb.as_retriever(search_kwargs={"k": 20})
    )

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""


    if 'vectordb' not in globals() and vectordb is None:
        return "No vector store available", []
    
    #prefer compression_retriever when available , otherwise use fallback to basic retriever
    try:
        if compression_retriever is not None:
            retrieved_docs = compression_retriever.invoke(
            query
        )
            
        else:
            retrieved_docs = vectordb.as_retriever(search_kwargs={"k": 5}).get_relevant_documents(query)
    # retrieved_docs = vectordb.similarity_search(query, k=2)
    # serialized = "\n\n".join(
    #     (f"Source: {doc.metadata}\n Page Content: {doc.page_content}")
    #     for doc in retrieved_docs
    # )
       
    except Exception as e:
            print("error in retrieve_context",e)

    serialized = "\n\n".join(
            f"Source: {d.metadata.get('source','unknown')} | Page: {d.metadata.get('page','?')}\n{d.page_content}"
            for d in retrieved_docs
        )
    return serialized, retrieved_docs

#### Finally chaining our retriver tool with LLM and prompt

Note: This our now the go to method for performing RAG instead of LCEL based chaining in Langchain since v1

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

# Instantiate the LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", streaming=True)
openAIClient = llm.client

tools = [retrieve_context]

# prompt = """You are an insurance policy QA assistant. Use ONLY the content returned by the retrieval tool(s) to answer — DO NOT rely on external knowledge or guess.

# REQUIRED BEHAVIOR:
# 1) ALWAYS respond EXACTLY in this two-line format and nothing else:
# answer: [your concise answer here]
# source: [filename.pdf | page X][; filename2.pdf | page Y]  # list one or more sources separated by semicolons

# 2) If the provided documents do NOT contain an answer, respond exactly:
# answer: I don't know — not found in provided documents.
# source: none

# 3) When you can, include a 1-2 sentence quoted excerpt that directly supports your answer. Put the excerpt inside double quotes on the same answer line after the main sentence, followed by the source line.

# 4) Use the retriever tool results only. Do not invent facts, numbers, policy limits, dates, or legal language. If uncertain, say so.

# 5) Format rules:
#    - Numeric amounts: use digits and currency symbol, e.g. $500,000
#    - Dates: YYYY-MM-DD
#    - Keep the answer concise (1-3 short sentences).
#    - Cite page numbers and source filename exactly as returned by the retriever metadata.

# EXAMPLES:
# Correct:
# answer: The policy provides up to $500,000 in life insurance coverage. "Coverage limit: $500,000 (see policy limits section)." 
# source: policy_document_v1.pdf | page 12

# If not found:
# answer: I don't know — not found in provided documents.
# source: none

# If user question is ambiguous:
# answer: Please clarify: do you mean coverage amount or eligibility criteria?
# source: none

# CRITICAL: Use only the retrieved document text. Do not include any additional commentary, operational notes, or tool output. End response after the two required lines."""



agent = create_agent(llm,
                     tools,
                     system_prompt=prompt,
                     middleware = [# Redact emails in user input before sending to model
                     PIIMiddleware(
                        "email",
                        strategy="redact",
                        apply_to_input=True,
                     ),
                     # Mask credit cards in user input
                     PIIMiddleware(
                        "credit_card",
                        strategy="mask",
                        apply_to_input=True,
                     ),
                     # Block API keys - raise error if detected
                     PIIMiddleware(
                        "api_key",
                        detector=r"sk-[a-zA-Z0-9]{32}",
                        strategy="block",
                        apply_to_input=True,
                     ),])


In [ ]:
def check_moderation_flag(expression):
    moderation_response = openAIClient.moderations.create(input=expression)
    flagged = moderation_response.results[0].flagged
    return flagged

In [ ]:
# ---------- 1) canonicalize text ----------
def canonicalize(text: str) -> str:
    # remove nulls, weird control chars, excessive whitespace, HTML comments, scripts
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<script.*?>.*?</script>', ' ', text, flags=re.DOTALL|re.IGNORECASE)
    text = text.replace('\x00', ' ')
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

# ---------- 2) basic prompt-injection signature detector (regex) ----------
INJECTION_PATTERNS = [
    r'(?i)ignore (?:previous|earlier) instructions',
    r'(?i)disregard (?:previous|earlier) instructions',
    r'(?i)follow these steps:',
    r'(?i)execute the following',
    r'(?i)now do exactly as follows',
    r'(?i)you are now',
    r'(?i)system message:',
    r'(?i)if you are reading this',
]

def detect_injection(text: str) -> List[str]:
    found = []
    for p in INJECTION_PATTERNS:
        if re.search(p, text):
            found.append(p)
    return found

#### Function that we will be using to answer the user query from our complete RAG system

In [ ]:
from langchain.messages import HumanMessage

def insurance_agent(query: str):
    query = canonicalize(query)
    injections = detect_injection(query)
    if injections:
        return f"Prompt injection detected! Patterns: {injections}"
    
    is_flagged = check_moderation_flag(query)

    if is_flagged:
        return "Input content flagged by moderation filter."

    response = agent.invoke({
        'messages': [
            HumanMessage(content=(
            query
            ))
        ]
    })

    print(response['messages'][-1].content)

In [35]:
print(prompt.messages[0].prompt.template)

You are an insurance policy QA assistant. Use ONLY the content returned by the retrieval tool(s) to answer — DO NOT rely on external knowledge or guess.

REQUIRED BEHAVIOR:
1) ALWAYS respond EXACTLY in this two-line format and nothing else:
answer: [your concise answer here]
source: [filename.pdf | page X][; filename2.pdf | page Y]  # list one or more sources separated by semicolons

2) If the provided documents do NOT contain an answer, respond exactly:
answer: I don't know — not found in provided documents.
source: none

3) When you can, include a 1-2 sentence quoted excerpt that directly supports your answer. Put the excerpt inside double quotes on the same answer line after the main sentence, followed by the source line.

4) Use the retriever tool results only. Do not invent facts, numbers, policy limits, dates, or legal language. If uncertain, say so.

5) Format rules:
   - Numeric amounts: use digits and currency symbol, e.g. $500,000
   - Dates: YYYY-MM-DD
   - Keep the answer c

In [24]:

insurance_agent( "What is the life insurance policy coverage amount?")

ValidationError: 21 validation errors for SystemMessage
content.str
  Input should be a valid string [type=string_type, input_value=ChatPromptTemplate(input_... additional_kwargs={})]), input_type=ChatPromptTemplate]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
content.list[union[str,dict[any,any]]].0.str
  Input should be a valid string [type=string_type, input_value=('name', None), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
content.list[union[str,dict[any,any]]].0.dict[any,any]
  Input should be a valid dictionary [type=dict_type, input_value=('name', None), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.12/v/dict_type
content.list[union[str,dict[any,any]]].1.str
  Input should be a valid string [type=string_type, input_value=('input_variables', ['question']), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
c

In [ ]:
# insurance_agent( "Can a 100 year plus person do a term insurance?")

2025-11-20 13:34:55,908 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-20 13:34:56,719 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-20 13:35:00,157 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-20 13:35:02,786 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-20 13:35:06,029 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: A 100-year-old individual is typically ineligible for term insurance, as policies usually have a maximum entry age limit. "The Policy shall however become void from commencement if the Age of the Life Assured at the Policy Commencement Date is found to be higher than the maximum… permissible under this Policy." 
source: HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf | page ?


In [ ]:
# insurance_agent("what is the Definitions of Critical Illnesses? based on policy?")

2025-11-20 13:35:32,923 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-20 13:35:33,666 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-20 13:35:36,876 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The definitions of critical illnesses include conditions such as myocardial infarction, which is characterized by heart muscle death due to inadequate blood supply. "Myocardial Infarction (First Heart Attack of specific severity) means the death of a portion of the heart muscle as a result of inadequate blood supply." 
source: HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf | page ?; HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | page ?


In [ ]:
# insurance_agent("what is the life insurance coverage for disability?")

2025-11-20 13:26:22,583 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-20 13:26:23,401 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-20 13:26:26,532 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


answer: The policy provides a lump sum benefit equal to 100% of the Sum Insured for specified critical illnesses, but does not explicitly mention coverage for disability under life insurance. 
source: Policy+Documents\HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | Page: ?


In [ ]:
# retrieve_context("what is the Definitions of Critical Illnesses? based on policy?")

In [ ]:
# retrieve_context("Can a 100 year plus person do a term insurance?")

In [ ]:
import pandas as pd
from phoenix.evals import *
from phoenix.evals import (
    OpenAIModel,
    HallucinationEvaluator,
    FaithfulnessEvaluator,
    AnswerCorrectnessEvaluator,
    RelevanceEvaluator,
    RetrievalEvaluator
)

In [ ]:
df = pd.DataFrame([
    {
        "question": "What is the coverage amount of the policy?",
        "answer": "The coverage is 10 lakhs.",
        "context": ["The policy document states a coverage of ₹10,00,000."],
        "ground_truth": "Coverage amount is 10 lakhs."
    }
])

questions = df["question"].tolist()
answers   = df["answer"].tolist()
contexts  = df["context"].tolist()
ground_truths = df["ground_truth"].tolist()

In [ ]:
retrieval_eval = evaluate_retrieval(
    dataset=df,
    llm_client=client,
    model="gpt-4o-mini",      # judge model
    include_metrics=True,     # computes Precision@K, Recall@K, NDCG, MRR
    k=5                       # Top-K to evaluate
)

print("RETRIEVAL METRICS")
print(retrieval_eval.metrics)

In [ ]:
retrieval_eval = RetrievalEvaluator(
    model=model,
    top_k=3
)

results_retrieval = retrieval_eval.evaluate(
    queries=questions,
    contexts=contexts,
)
print(results_retrieval)

In [ ]:
relevance_eval = RelevanceEvaluator(
    model=model
)

results_relevance = relevance_eval.evaluate(
    queries=questions,
    responses=answers
)

print(results_relevance)

In [ ]:
faithfulness_eval = FaithfulnessEvaluator(
    model=model
)

results_faithfulness = faithfulness_eval.evaluate(
    contexts=contexts,
    responses=answers
)

print(results_faithfulness)

In [ ]:
hallucination_eval = HallucinationEvaluator(
    model=model
)

results_hallucination = hallucination_eval.evaluate(
    queries=questions,
    contexts=contexts,
    responses=answers
)

print(results_hallucination)

In [ ]:
correctness_eval = AnswerCorrectnessEvaluator(
    model=model
)

results_correctness = correctness_eval.evaluate(
    ground_truths=ground_truths,
    responses=answers
)

print(results_correctness)

In [ ]:
from phoenix.evals.experiment import Experiment

experiment = Experiment(
    evaluators=[
        RetrievalEvaluator(model),
        FaithfulnessEvaluator(model),
        HallucinationEvaluator(model),
        AnswerCorrectnessEvaluator(model),
        RelevanceEvaluator(model),
    ]
)

experiment_results = experiment.run(
    queries=questions,
    contexts=contexts,
    responses=answers,
    ground_truths=ground_truths
)

print(experiment_results.to_pandas())